# Lending Club Loan Performance & Risk Analysis (2007–2018)

## Introduction

**Credit risk** and **portfolio profitability** go hand in hand in P2P lending: mispriced or poorly managed risk drives up defaults and erodes investor confidence.

This notebook is the **business-query** step of the pipeline. We use **SQL** on the cleaned Lending Club data (2007–2018) in `cleaned_data.db` to answer questions such as:

- **How often do loans default overall**, and how does default risk vary by grade, income, loan size, and FICO?
- **Which segments are most profitable** after accounting for defaults?
- **How has default risk evolved** across issue years (vintages)?

Results from these queries feed **Power BI** dashboards and support underwriting, pricing, and portfolio decisions.

---

## Contents

This notebook covers:

1. **Dataset connection and overview** — Connect to `cleaned_data.db`, preview the `loans` table, and review key fields.
2. **Default rates** — Default rates overall and by grade, income, loan size, FICO, and vintage.
3. **Interest** — Interest income and related metrics by segment.
4. **Loan purpose** — Default rate, volume, and profit by stated use.
5. **Term and affordability (DTI)** — Default rate and volume by term and DTI band.
6. **Additional risk views** — Further risk and performance perspectives for reporting and dashboards.

---

## Dataset connection and overview

In this section we connect to the cleaned SQLite database (`cleaned_data.db`) and preview the `loans` table, which is the main fact table used throughout the analysis.

Some of the most important fields we will rely on are:

- **`grade`**: Lending Club credit grade assigned to the loan
- **`default`**: binary loan outcome (1 = defaulted, 0 = non-default)
- **`loan_amnt`**: original loan amount funded
- **`int_rate`**: interest rate charged on the loan
- **`annual_inc`**: borrower’s annual income
- **`dti`**: debt-to-income ratio at origination
- **`purpose`**: declared purpose for the loan (e.g., debt consolidation, business)
- **`fico_score`**: borrower’s FICO credit score
- **`issue_d`**: date the loan was issued

The query below simply pulls a small sample of rows so we can visually confirm the structure and values before computing any metrics.

In [15]:
%load_ext sql

%sql sqlite:///../data/cleaned_data.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [16]:
%%sql
SELECT *
FROM loans
LIMIT 5;

 * sqlite:///../data/cleaned_data.db
Done.


grade,delinq_2yrs,inq_last_6mths,dti,annual_inc,revol_util,loan_amnt,int_rate,term,issue_d,installment,earliest_cr_line,total_acc,open_acc,mths_since_last_delinq,total_pymnt,default,purpose_group,fico_score
C,0,1,5.91,55000.0,29.7,3600.0,13.99,36,2015-12-01 00:00:00.000000,123.03,2003-08-01 00:00:00.000000,13,7,30,4421.72,0,Debt,677.0
C,1,4,16.06,65000.0,19.2,24700.0,11.99,36,2015-12-01 00:00:00.000000,820.28,1999-12-01 00:00:00.000000,38,22,6,25679.66,0,Business,717.0
B,0,0,10.78,63000.0,56.2,20000.0,10.78,60,2015-12-01 00:00:00.000000,432.66,2000-08-01 00:00:00.000000,18,6,0,22705.92,0,Home,697.0
F,1,3,25.37,104433.0,64.5,10400.0,22.45,60,2015-12-01 00:00:00.000000,289.91,1998-06-01 00:00:00.000000,35,12,12,11740.5,0,Consumer,697.0
C,0,0,10.2,34000.0,68.4,11950.0,13.44,36,2015-12-01 00:00:00.000000,405.18,1987-10-01 00:00:00.000000,6,5,0,13708.95,0,Debt,692.0


---

## Default rate

We quantify **how often loans default** at the portfolio level and across borrower and loan segments. These metrics support risk-based pricing, exposure limits, and later profitability analysis.

**Metrics in this section:**

- **Portfolio default rate** — Overall default rate and volume.
- **By credit grade** — Default rate from A (lowest risk) to G (highest risk).
- **By FICO band** — Risk by credit-score tier (Poor / Fair / Good / Excellent).
- **By income level** — Default rate by borrower income band.
- **By loan size** — Risk by ticket size (Small / Medium / Large).

### Total default rate

In [17]:
%%sql
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate_pct
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_loans,total_defaults,default_rate_pct
1371165,294415,21.47


**Takeaway:** The portfolio default rate is **21.5%** (about one in five loans). With 294k+ defaults out of 1.37M loans, segment-level monitoring and risk-based pricing are essential. The breakdowns below show where risk is concentrated and where to tighten underwriting or pricing.

### Default Rate by Credit Grade

In [18]:
%%sql
SELECT 
    grade,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*), 2) AS default_rate_pct,
    ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,loans,defaults,default_rate_pct,pct_of_portfolio
A,236758,15869,6.7,17.3
B,398528,58357,14.64,29.1
C,390724,94687,24.23,28.5
D,206696,66797,32.32,15.1
E,96224,38609,40.12,7.0
F,32828,15261,46.49,2.4
G,9407,4835,51.4,0.7


**Takeaway:** Default risk increases sharply with grade (A: 6.7% → G: 51.4%). Grades B and C hold the largest share of the book; subprime grades (D–G) contribute disproportionately to defaults. Grade should drive pricing, credit policy, and exposure limits.

### Default Rate by FICO Score

In [19]:
%%sql
SELECT 
    CASE
        WHEN fico_score < 600 THEN 'Poor'
        WHEN fico_score < 700 THEN 'Fair'
        WHEN fico_score < 750 THEN 'Good'
        ELSE 'Excellent'
    END AS credit_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY credit_group
ORDER BY CASE credit_group
    WHEN 'Poor' THEN 1 WHEN 'Fair' THEN 2 WHEN 'Good' THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate_pct
Fair,837712,210540,25.13
Good,425361,72911,17.14
Excellent,108092,10964,10.14


**Takeaway:** Default rate is lowest for **Excellent** FICO (10.1%) and highest for **Fair** (25.1%). Fair is the largest segment and drives most defaults. The step-up from Good to Fair is large; underwriting and limits for Fair borrowers should be a priority.

### Default Rate by Income Level

In [20]:
%%sql
SELECT 
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY income_group
ORDER BY CASE income_group
    WHEN 'Low Income' THEN 1 WHEN 'Middle Income' THEN 2 ELSE 3 END;

 * sqlite:///../data/cleaned_data.db
Done.


income_group,loans,defaults,default_rate_pct
Low Income,214632,54213,25.26
Middle Income,675851,151339,22.39
High Income,480682,88863,18.49


**Takeaway:** Default rate is highest for **Low** income (25.3%) and lowest for **High** (18.5%). The spread is smaller than for grade or FICO; income is useful when combined with DTI and credit quality for affordability and risk assessment.

### Default Rate by Loan Amount

In [21]:
%%sql
SELECT 
    CASE
        WHEN loan_amnt < 5000 THEN 'Small'
        WHEN loan_amnt < 15000 THEN 'Medium'
        ELSE 'Large'
    END AS loan_size,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate

FROM loans
GROUP BY loan_size;

 * sqlite:///../data/cleaned_data.db
Done.


loan_size,loans,defaults,default_rate
Large,590009,144248,24.45
Medium,646277,127803,19.78
Small,134879,22364,16.58


**Takeaway:** Default rate increases with loan size (Small: 16.6% → Large: 24.5%). Larger tickets carry both higher default risk and higher exposure per loan. Tighter underwriting or risk-based pricing for large loans can improve portfolio outcomes.

---

## Loan purpose

Loan **purpose** drives both demand and risk: debt consolidation, home improvement, and business use have different default profiles and portfolio implications. Segmenting by purpose helps set product-level limits and pricing.

**Metrics in this section:**

- **Default rate by purpose** — Risk by stated use (e.g. Debt, Home, Business, Consumer).
- **Profit by purpose** — Profit or loss by purpose to see which uses are profitable.

### Default rate by loan purpose

In [ ]:
%%sql
SELECT 
    purpose_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY purpose_group
ORDER BY default_rate_pct DESC;

 * sqlite:///../data/cleaned_data.db
Done.


purpose_group,loans,defaults,default_rate_pct,pct_of_portfolio
Business,16777,5229,31.17,1.2
Health/Education,16225,3788,23.35,1.2
Other,79791,18301,22.94,5.8
Debt,1095457,234854,21.44,79.9
Consumer,66159,13155,19.88,4.8
Home,96756,19088,19.73,7.1


**Takeaway:** Default rates vary meaningfully by purpose. Higher-risk purposes (e.g. Business, Consumer) warrant tighter limits or higher pricing; dominant segments (e.g. Debt) drive most of the book and should be monitored closely.

### Profit by purpose

In [24]:
%%sql
SELECT 
    purpose_group,
    COUNT(*) AS loans,
    ROUND(SUM(total_pymnt - loan_amnt), 2) AS profit
FROM loans
GROUP BY purpose_group
ORDER BY profit DESC;

 * sqlite:///../data/cleaned_data.db
Done.


purpose_group,loans,profit
Debt,1095457,369410930.99
Home,96756,8082456.83
Health/Education,16225,-3968739.7
Business,16777,-11532307.34
Consumer,66159,-11882574.88
Other,79791,-12757134.69


**Takeaway:** Profit by purpose identifies which uses add or destroy value. Combine with default rates to balance growth and risk by product.

---

## Term and affordability (DTI)

**Term** (36 vs 60 months) and **debt-to-income (DTI)** are key levers: longer terms increase exposure and often risk; higher DTI indicates less capacity to absorb shocks. Segmenting by term and DTI supports underwriting and pricing.

**Metrics in this section:**

- **Default rate by term** — Risk for 36- vs 60-month loans.
- **Default rate by DTI band** — Risk by affordability tier (Low / Medium / High / Very high DTI).
- **Volume by term** — Share of portfolio in each term to assess concentration.

### Default rate by term

In [25]:
%%sql
SELECT 
    term,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY term
ORDER BY term;

 * sqlite:///../data/cleaned_data.db
Done.


term,loans,defaults,default_rate_pct,pct_of_portfolio
36,1035787,178296,17.21,75.5
60,335378,116119,34.62,24.5


**Takeaway:** 60-month loans often show higher default rates than 36-month; longer terms mean more time for distress. This can be used to set term-based pricing and limits.

### Default rate by DTI band

In [26]:
%%sql
SELECT 
    CASE 
        WHEN dti < 10 THEN "Low"
        WHEN dti < 20 THEN "Medium"
        WHEN dti < 30 THEN "High"
        ELSE "Very high"
    END AS dti_band,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY dti_band
ORDER BY CASE dti_band 
    WHEN "Low" THEN 1 WHEN "Medium" THEN 2 WHEN "High" THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


dti_band,loans,defaults,default_rate_pct
Low,250260,41120,16.43
Medium,571897,109914,19.22
High,416700,102223,24.53
Very high,132308,41158,31.11


**Takeaway:** Higher DTI is associated with higher default risk. Use DTI bands in underwriting and pricing; cap or price up high-DTI segments.

---